In [68]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName("Working with Join and Window")\
.getOrCreate()
spark

In [69]:
import pandas as pd

data = {
    "patient_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "patient_name": ["Rahul Sharma", "Priya Reddy", "Amit Kumar", "Sneha Patel","Farhan Ali", "Neha Singh", "Arjun Verma", "Meera Nair","Kiran Rao", "Nisha Reddy"],
    "city": ["Hyderabad", "Bangalore", "Mumbai", "Chennai","Delhi", None, "Pune", "Kochi","Hyderabad", "Bangalore"],
    "age": [35, 29, 42, 31, 55, 38, 26, 48, 33, 41],
    "gender": ["Male", "Female", "Male", "Female","Male", "Female", "Male", "Female","Male", "Female"],
    "blood_group": ["O+", "A+", "B+", "O+","AB+", "A+", "B+", "O-",None, "A+"],
    "insurance_status": ["Active", "Active", "Inactive", "Active","Active", "Inactive", "Active", "Active","Inactive", "Active"]
}

patients_df = pd.DataFrame(data)
patients_df.to_csv("patients.csv", index=False)
print("patients.csv created successfully!")

patients.csv created successfully!


In [70]:
import pandas as pd

data = {
    "appointment_id": [5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010],
    "patient_id": [101, 102, 101, 103, 104, 105, 107, 110, 120, 108],
    "doctor_name": ["Dr. Ramesh", "Dr. Suresh", "Dr. Anita", "Dr. Ramesh", "Dr. Priya", "Dr. Anita", "Dr. Suresh", "Dr. Priya", "Dr. Ramesh", "Dr. Anita"],
    "department": ["Cardiology", "Neurology", "Dermatology", "Cardiology", "Orthopedics", "Dermatology", "Neurology", "Orthopedics", "Cardiology", "Dermatology"],
    "appointment_date": ["2025-01-10", "2025-01-11", "2025-01-15", "2025-01-20", "2025-01-22", "2025-01-25", "2025-02-01", "2025-02-03", "2025-02-05", "2025-02-10"],
    "consultation_fee": [1500, 2000, 1000, 1500, 2500, 1000, 2000, 2500, 1500, None],
    "status": ["Completed", "Completed", "Completed", "Cancelled", "Completed", "Pending", "Completed", "Completed", "Completed", "Pending"]
}

appointments_df = pd.DataFrame(data)
appointments_df.to_csv("appointments.csv", index=False)
print("appointments.csv created successfully!")

appointments.csv created successfully!


In [71]:
import json

data = [
    {
        "patient_id": 101,
        "preferred_hospital": "Apollo",
        "contact": {
            "phone": "9876500011",
            "email": "rahul@gmail.com"
        }
    },
    {
        "patient_id": 102,
        "preferred_hospital": "Yashoda",
        "contact": {
            "phone": None,
            "email": "priya@gmail.com"
        }
    },
    {
        "patient_id": 103,
        "preferred_hospital": "Care",
        "contact": {
            "phone": "9876500013",
            "email": None
        }
    },
    {
        "patient_id": 104,
        "preferred_hospital": None,
        "contact": {
            "phone": "9876500014",
            "email": "sneha@gmail.com"
        }
    }
]

with open("patient_preferences.json", "w") as f:
    json.dump(data, f, indent=4)

print("patient_preferences.json created successfully!")

patient_preferences.json created successfully!


**CSV Ingestion**

In [72]:
#1
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
patients.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [73]:
#2
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
appointments.show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|          2000.0|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|          1500.0|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|          2500.0|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|          1000.0|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|          2000.0|Completed|
|          5008|       110|  Dr. Priya|O

In [74]:
#3
patients.printSchema()
appointments.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- consultation_fee: double (nullable = true)
 |-- status: string (nullable = true)



In [75]:
#4
print(patients.count())
print(appointments.count())


10
10


In [76]:
#5
patients.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows


In [77]:
#6
patients.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [78]:
#7
appointments.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [79]:
#8
patients.write.mode("overwrite").parquet("patients_parquet")

In [80]:
#9
patients_parquet = spark.read.parquet("patients_parquet")
patients_parquet.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [81]:
#10
print("Csv Records counts:",patients.count())
print("Parquet Records counts:",patients_parquet.count())

Csv Records counts: 10
Parquet Records counts: 10


**Filtering**

In [82]:
#11
patients.filter(patients.city == "Hyderabad").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [83]:
#12
patients.filter(patients.gender == "Female").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [84]:
#13
patients.filter(patients.age > 40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [85]:
#14
appointments.filter(appointments.status == "Completed").show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|          2000.0|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|          2500.0|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|          2000.0|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|          2500.0|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|          1500.0|Completed|
+--------------+----------+-----------+-

In [86]:
#15
appointments.filter(appointments.status == "Pending").show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|          1000.0|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [87]:
#16
appointments.filter(appointments.consultation_fee > 1500).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|          2000.0|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|          2500.0|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|          2000.0|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|          2500.0|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [88]:
#17
patients.filter(patients.insurance_status == "Active").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [89]:
#18
patients.filter(patients.insurance_status == "Inactive").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [90]:
#19
patients.filter(patients.blood_group == "O+").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [91]:
#20
appointments.filter(appointments.department == "Cardiology").show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|          1500.0|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|          1500.0|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|          1500.0|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



**Null Handling**

In [92]:
#21
from pyspark.sql.functions import *

patients.filter(col("city").isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [93]:
#22
patients.filter(col("blood_group").isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [94]:
#23
appointments.filter(col("consultation_fee").isNull()).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [96]:
#24
patients.select([count(when(col(c).isNull(), c)).alias(c) for c in patients.columns]).show()
appointments.select([count(when(col(c).isNull(), c)).alias(c) for c in appointments.columns]).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+

+--------------+----------+-----------+----------+----------------+----------------+------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|status|
+--------------+----------+-----------+----------+----------------+----------------+------+
|             0|         0|          0|         0|               0|               1|     0|
+--------------+----------+-----------+----------+----------------+----------------+------+



In [98]:
#25
patients.na.fill({"city": "Unknown"}).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [99]:
#26
patients.na.fill({"blood_group": "Not Available"}).show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [100]:
#27
appointments.fillna({"consultation_fee": 0}).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|          2000.0|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|          1500.0|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|          2500.0|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|          1000.0|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|          2000.0|Completed|
|          5008|       110|  Dr. Priya|O

In [101]:
#28
appointments.dropna(subset=["consultation_fee"]).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|          2000.0|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|          1500.0|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|          2500.0|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|          1000.0|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|          2000.0|Completed|
|          5008|       110|  Dr. Priya|O

In [103]:
#29
patients = patients.withColumn("data_quality_status",when(col("city").isNull() | col("blood_group").isNull(),"Incomplete").otherwise("Complete"))
patients.show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|         Incomplete|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|           Complete|
|       108|  Meera Nair|    Kochi| 48|F

In [104]:
#30
patients.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    8|
|         Incomplete|    2|
+-------------------+-----+



**Built-in Functions**

In [109]:
#31
patients.select(upper("patient_name")).show()

+-------------------+
|upper(patient_name)|
+-------------------+
|       RAHUL SHARMA|
|        PRIYA REDDY|
|         AMIT KUMAR|
|        SNEHA PATEL|
|         FARHAN ALI|
|         NEHA SINGH|
|        ARJUN VERMA|
|         MEERA NAIR|
|          KIRAN RAO|
|        NISHA REDDY|
+-------------------+



In [110]:
#32
patients.select(lower("patient_name")).show()

+-------------------+
|lower(patient_name)|
+-------------------+
|       rahul sharma|
|        priya reddy|
|         amit kumar|
|        sneha patel|
|         farhan ali|
|         neha singh|
|        arjun verma|
|         meera nair|
|          kiran rao|
|        nisha reddy|
+-------------------+



In [113]:
#33
patients.select("patient_name",length("patient_name").alias("name_length")).show()

+------------+-----------+
|patient_name|name_length|
+------------+-----------+
|Rahul Sharma|         12|
| Priya Reddy|         11|
|  Amit Kumar|         10|
| Sneha Patel|         11|
|  Farhan Ali|         10|
|  Neha Singh|         10|
| Arjun Verma|         11|
|  Meera Nair|         10|
|   Kiran Rao|          9|
| Nisha Reddy|         11|
+------------+-----------+



In [114]:
#34
patients.select("patient_name",substring("patient_name",1,3).alias("first_three_letters")).show()

+------------+-------------------+
|patient_name|first_three_letters|
+------------+-------------------+
|Rahul Sharma|                Rah|
| Priya Reddy|                Pri|
|  Amit Kumar|                Ami|
| Sneha Patel|                Sne|
|  Farhan Ali|                Far|
|  Neha Singh|                Neh|
| Arjun Verma|                Arj|
|  Meera Nair|                Mee|
|   Kiran Rao|                Kir|
| Nisha Reddy|                Nis|
+------------+-------------------+



In [116]:
#35
patients = patients.withColumn("age_group",when(col("age") < 30, "Young")
.when(col("age") < 50, "Adult")
.otherwise("Senior"))
patients.show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|    Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|    Adult|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|    Adult|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|   Senior|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|         Incomplete|    Adult|
|       107| Arjun Verma|     Pune| 26|  Male|

In [118]:
#36
patients = patients.withColumn("insurance_flag",when(col("insurance_status") == "Active",1).otherwise(0))
patients.show()


+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|    Young|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|    Adult|             0|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|    Adult|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|   Senior|             1|
|       106|  Neha Singh|     NU

In [120]:
#37
patients = patients.withColumn("senior_citizen",when(col("age") >= 50,1).otherwise(0))
patients.show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|    Young|             1|             0|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|    Adult|             0|             0|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|    Adult|             1|             0|
|       105|  Farhan Ali|    Delhi| 55|  Male|  

In [126]:
#38
patients.select(concat_ws(" - ","patient_name","city").alias("name_city")).show(truncate=False)

+------------------------+
|name_city               |
+------------------------+
|Rahul Sharma - Hyderabad|
|Priya Reddy - Bangalore |
|Amit Kumar - Mumbai     |
|Sneha Patel - Chennai   |
|Farhan Ali - Delhi      |
|Neha Singh              |
|Arjun Verma - Pune      |
|Meera Nair - Kochi      |
|Kiran Rao - Hyderabad   |
|Nisha Reddy - Bangalore |
+------------------------+



In [127]:
#39
patients.select(trim("patient_name")).show()

+------------------+
|trim(patient_name)|
+------------------+
|      Rahul Sharma|
|       Priya Reddy|
|        Amit Kumar|
|       Sneha Patel|
|        Farhan Ali|
|        Neha Singh|
|       Arjun Verma|
|        Meera Nair|
|         Kiran Rao|
|       Nisha Reddy|
+------------------+



In [129]:
#40
patients.select(upper("city")).show()

+-----------+
|upper(city)|
+-----------+
|  HYDERABAD|
|  BANGALORE|
|     MUMBAI|
|    CHENNAI|
|      DELHI|
|       NULL|
|       PUNE|
|      KOCHI|
|  HYDERABAD|
|  BANGALORE|
+-----------+



**GroupBy and Aggregations**

In [130]:
#41
patients.groupby("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [131]:
#42
patients.groupby("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [132]:
#43
patients.groupby("blood_group").count().show()

+-----------+-----+
|blood_group|count|
+-----------+-----+
|        AB+|    1|
|       NULL|    1|
|         O+|    2|
|         O-|    1|
|         B+|    2|
|         A+|    3|
+-----------+-----+



In [133]:
#44
appointments.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [134]:
#45
patients.groupBy("city").agg(avg("age").alias("average_age")).show()

+---------+-----------+
|     city|average_age|
+---------+-----------+
|Bangalore|       35.0|
|    Kochi|       48.0|
|  Chennai|       31.0|
|     NULL|       38.0|
|   Mumbai|       42.0|
|     Pune|       26.0|
|    Delhi|       55.0|
|Hyderabad|       34.0|
+---------+-----------+



In [135]:
#46
patients.groupBy("city").agg(max("age").alias("maximum_age")).show()

+---------+-----------+
|     city|maximum_age|
+---------+-----------+
|Bangalore|         41|
|    Kochi|         48|
|  Chennai|         31|
|     NULL|         38|
|   Mumbai|         42|
|     Pune|         26|
|    Delhi|         55|
|Hyderabad|         35|
+---------+-----------+



In [136]:
#47
patients.groupBy("city").agg(min("age").alias("minimum_age")).show()

+---------+-----------+
|     city|minimum_age|
+---------+-----------+
|Bangalore|         29|
|    Kochi|         48|
|  Chennai|         31|
|     NULL|         38|
|   Mumbai|         42|
|     Pune|         26|
|    Delhi|         55|
|Hyderabad|         33|
+---------+-----------+



In [137]:
#48
appointments.groupBy("department").agg(avg("consultation_fee").alias("average_consultation_fee")).show()

+-----------+------------------------+
| department|average_consultation_fee|
+-----------+------------------------+
|  Neurology|                  2000.0|
|Dermatology|                  1000.0|
| Cardiology|                  1500.0|
|Orthopedics|                  2500.0|
+-----------+------------------------+



In [138]:
#49
appointments.groupBy("department").agg(sum("consultation_fee").alias("total_consultation_fee")).show()

+-----------+----------------------+
| department|total_consultation_fee|
+-----------+----------------------+
|  Neurology|                4000.0|
|Dermatology|                2000.0|
| Cardiology|                4500.0|
|Orthopedics|                5000.0|
+-----------+----------------------+



In [144]:
#50
appointments.groupBy("department").agg(sum("consultation_fee").alias("revenue")).orderBy(desc("revenue")).show(1)

+-----------+-------+
| department|revenue|
+-----------+-------+
|Orthopedics| 5000.0|
+-----------+-------+
only showing top 1 row


**Joins**

In [145]:
#51
patients.join(appointments,"patient_id","inner").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|          5003|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|    

In [146]:
#52
patients.join(appointments,"patient_id","left").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|          5003|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|    

In [147]:
#53
patients.join(appointments,"patient_id","right").show()

+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|          5001| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|           Complete|    Young|             1

In [148]:
#54
patients.join(appointments,"patient_id","full").show()

+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|          5001| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|    Adult|             1

In [149]:
#55
patients.join(appointments,"patient_id","left_anti").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|         Incomplete|    Adult|             0|             0|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|         Incomplete|    Adult|             0|             0|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+



In [150]:
#56
appointments.join(patients,"patient_id","left_anti").show()

+----------+--------------+-----------+----------+----------------+----------------+---------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|
+----------+--------------+-----------+----------+----------------+----------------+---------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|          1500.0|Completed|
+----------+--------------+-----------+----------+----------------+----------------+---------+



In [151]:
#57
appointments.groupBy("patient_id").count().show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [153]:
#58
patients.join(appointments,"patient_id").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_paid")).show()

+----------+------------+----------+
|patient_id|patient_name|total_paid|
+----------+------------+----------+
|       107| Arjun Verma|    2000.0|
|       108|  Meera Nair|      NULL|
|       110| Nisha Reddy|    2500.0|
|       105|  Farhan Ali|    1000.0|
|       101|Rahul Sharma|    2500.0|
|       104| Sneha Patel|    2500.0|
|       103|  Amit Kumar|    1500.0|
|       102| Priya Reddy|    2000.0|
+----------+------------+----------+



In [154]:
#59
patients.join(appointments,"patient_id").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_paid")).orderBy(desc("total_paid")).show(1)

+----------+------------+----------+
|patient_id|patient_name|total_paid|
+----------+------------+----------+
|       101|Rahul Sharma|    2500.0|
+----------+------------+----------+
only showing top 1 row


In [155]:
#60
patients.join(appointments,"patient_id").groupBy("patient_id","patient_name").count().show()

+----------+------------+-----+
|patient_id|patient_name|count|
+----------+------------+-----+
|       107| Arjun Verma|    1|
|       108|  Meera Nair|    1|
|       110| Nisha Reddy|    1|
|       105|  Farhan Ali|    1|
|       101|Rahul Sharma|    2|
|       104| Sneha Patel|    1|
|       103|  Amit Kumar|    1|
|       102| Priya Reddy|    1|
+----------+------------+-----+



**Window Functions**

In [159]:
#61
from pyspark.sql.window import Window

df = patients.join(appointments,"patient_id").groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("total_fee"))
w = Window.orderBy(col("total_fee").desc())
df.withColumn("rank",rank().over(w)).show()

+----------+------------+---------+----+
|patient_id|patient_name|total_fee|rank|
+----------+------------+---------+----+
|       110| Nisha Reddy|   2500.0|   1|
|       101|Rahul Sharma|   2500.0|   1|
|       104| Sneha Patel|   2500.0|   1|
|       107| Arjun Verma|   2000.0|   4|
|       102| Priya Reddy|   2000.0|   4|
|       103|  Amit Kumar|   1500.0|   6|
|       105|  Farhan Ali|   1000.0|   7|
|       108|  Meera Nair|     NULL|   8|
+----------+------------+---------+----+



In [160]:
#62
w = Window.orderBy(col("total_fee").desc())
df.withColumn("dense_rank",dense_rank().over(w)).show()

+----------+------------+---------+----------+
|patient_id|patient_name|total_fee|dense_rank|
+----------+------------+---------+----------+
|       110| Nisha Reddy|   2500.0|         1|
|       101|Rahul Sharma|   2500.0|         1|
|       104| Sneha Patel|   2500.0|         1|
|       107| Arjun Verma|   2000.0|         2|
|       102| Priya Reddy|   2000.0|         2|
|       103|  Amit Kumar|   1500.0|         3|
|       105|  Farhan Ali|   1000.0|         4|
|       108|  Meera Nair|     NULL|         5|
+----------+------------+---------+----------+



In [164]:
#63
df.withColumn("row_number",row_number().over(w)).show()

+----------+------------+---------+----------+
|patient_id|patient_name|total_fee|row_number|
+----------+------------+---------+----------+
|       110| Nisha Reddy|   2500.0|         1|
|       101|Rahul Sharma|   2500.0|         2|
|       104| Sneha Patel|   2500.0|         3|
|       107| Arjun Verma|   2000.0|         4|
|       102| Priya Reddy|   2000.0|         5|
|       103|  Amit Kumar|   1500.0|         6|
|       105|  Farhan Ali|   1000.0|         7|
|       108|  Meera Nair|     NULL|         8|
+----------+------------+---------+----------+



In [165]:
#64
df.orderBy(col("total_fee").desc()).show(1)

+----------+------------+---------+
|patient_id|patient_name|total_fee|
+----------+------------+---------+
|       101|Rahul Sharma|   2500.0|
+----------+------------+---------+
only showing top 1 row


In [167]:
#65
df.orderBy(desc("total_fee")).show(3)

+----------+------------+---------+
|patient_id|patient_name|total_fee|
+----------+------------+---------+
|       101|Rahul Sharma|   2500.0|
|       110| Nisha Reddy|   2500.0|
|       104| Sneha Patel|   2500.0|
+----------+------------+---------+
only showing top 3 rows


In [170]:
#66
city_df = patients.join(appointments,"patient_id").groupBy("city","patient_name").agg(sum("consultation_fee").alias("total_fee"))
w1 = Window.partitionBy("city").orderBy(desc("total_fee"))
city_df.withColumn("rank",rank().over(w1)).filter("rank=1").show()

+---------+------------+---------+----+
|     city|patient_name|total_fee|rank|
+---------+------------+---------+----+
|Bangalore| Nisha Reddy|   2500.0|   1|
|  Chennai| Sneha Patel|   2500.0|   1|
|    Delhi|  Farhan Ali|   1000.0|   1|
|Hyderabad|Rahul Sharma|   2500.0|   1|
|    Kochi|  Meera Nair|     NULL|   1|
|   Mumbai|  Amit Kumar|   1500.0|   1|
|     Pune| Arjun Verma|   2000.0|   1|
+---------+------------+---------+----+



In [177]:
#67
w2 = Window.partitionBy("city").orderBy("total_fee")
city_df.withColumn("rank",rank().over(w2)).filter("rank=1").show()

+---------+------------+---------+----+
|     city|patient_name|total_fee|rank|
+---------+------------+---------+----+
|Bangalore| Priya Reddy|   2000.0|   1|
|  Chennai| Sneha Patel|   2500.0|   1|
|    Delhi|  Farhan Ali|   1000.0|   1|
|Hyderabad|Rahul Sharma|   2500.0|   1|
|    Kochi|  Meera Nair|     NULL|   1|
|   Mumbai|  Amit Kumar|   1500.0|   1|
|     Pune| Arjun Verma|   2000.0|   1|
+---------+------------+---------+----+



In [176]:
#68
df.withColumn("running_total",sum("total_fee").over(Window.orderBy("total_fee").rowsBetween(Window.unboundedPreceding,0))).show()

+----------+------------+---------+-------------+
|patient_id|patient_name|total_fee|running_total|
+----------+------------+---------+-------------+
|       108|  Meera Nair|     NULL|         NULL|
|       105|  Farhan Ali|   1000.0|       1000.0|
|       103|  Amit Kumar|   1500.0|       2500.0|
|       107| Arjun Verma|   2000.0|       4500.0|
|       102| Priya Reddy|   2000.0|       6500.0|
|       110| Nisha Reddy|   2500.0|       9000.0|
|       101|Rahul Sharma|   2500.0|      11500.0|
|       104| Sneha Patel|   2500.0|      14000.0|
+----------+------------+---------+-------------+



In [180]:
#69
w = Window.orderBy(col("total_fee").desc())
df.withColumn("lead_fee",lead("total_fee").over(w)).show()

+----------+------------+---------+--------+
|patient_id|patient_name|total_fee|lead_fee|
+----------+------------+---------+--------+
|       110| Nisha Reddy|   2500.0|  2500.0|
|       101|Rahul Sharma|   2500.0|  2500.0|
|       104| Sneha Patel|   2500.0|  2000.0|
|       107| Arjun Verma|   2000.0|  2000.0|
|       102| Priya Reddy|   2000.0|  1500.0|
|       103|  Amit Kumar|   1500.0|  1000.0|
|       105|  Farhan Ali|   1000.0|    NULL|
|       108|  Meera Nair|     NULL|    NULL|
+----------+------------+---------+--------+



In [181]:
#70
df.withColumn("lag_fee",lag("total_fee").over(w)).show()

+----------+------------+---------+-------+
|patient_id|patient_name|total_fee|lag_fee|
+----------+------------+---------+-------+
|       110| Nisha Reddy|   2500.0|   NULL|
|       101|Rahul Sharma|   2500.0| 2500.0|
|       104| Sneha Patel|   2500.0| 2500.0|
|       107| Arjun Verma|   2000.0| 2500.0|
|       102| Priya Reddy|   2000.0| 2000.0|
|       103|  Amit Kumar|   1500.0| 2000.0|
|       105|  Farhan Ali|   1000.0| 1500.0|
|       108|  Meera Nair|     NULL| 1000.0|
+----------+------------+---------+-------+



**JSON Processing**

In [200]:
#71
preferences = spark.read.option("multiline",True).json("patient_preferences.json")
preferences.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [201]:
#72
preferences.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [202]:
#73
preferences.select("patient_id",col("contact.phone").alias("phone")).show()

+----------+----------+
|patient_id|     phone|
+----------+----------+
|       101|9876500011|
|       102|      NULL|
|       103|9876500013|
|       104|9876500014|
+----------+----------+



In [203]:
#74
preferences.select("patient_id",col("contact.email").alias("email")).show()

+----------+---------------+
|patient_id|          email|
+----------+---------------+
|       101|rahul@gmail.com|
|       102|priya@gmail.com|
|       103|           NULL|
|       104|sneha@gmail.com|
+----------+---------------+



In [204]:
#75
preferences.filter(col("contact.phone").isNull()).show(truncate=False)

+-----------------------+----------+------------------+
|contact                |patient_id|preferred_hospital|
+-----------------------+----------+------------------+
|{priya@gmail.com, NULL}|102       |Yashoda           |
+-----------------------+----------+------------------+



In [205]:
#76
preferences.filter(col("contact.email").isNull()).show(truncate=False)

+------------------+----------+------------------+
|contact           |patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|103       |Care              |
+------------------+----------+------------------+



In [206]:
#77
preferences.filter(col("preferred_hospital").isNull()).show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [210]:
#78
preferences = preferences.withColumn("phone",when(col("contact.phone").isNull(),"Not Available").otherwise(col("contact.phone")))
preferences.show(truncate=False)

+-----------------------------+----------+------------------+-------------+
|contact                      |patient_id|preferred_hospital|phone        |
+-----------------------------+----------+------------------+-------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |9876500011   |
|{priya@gmail.com, NULL}      |102       |Yashoda           |Not Available|
|{NULL, 9876500013}           |103       |Care              |9876500013   |
|{sneha@gmail.com, 9876500014}|104       |NULL              |9876500014   |
+-----------------------------+----------+------------------+-------------+



In [211]:
#79
preferences = preferences.withColumn("email",when(col("contact.email").isNull(),"Not Available").otherwise(col("contact.email")))
preferences.show(truncate=False)

+-----------------------------+----------+------------------+-------------+---------------+
|contact                      |patient_id|preferred_hospital|phone        |email          |
+-----------------------------+----------+------------------+-------------+---------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |9876500011   |rahul@gmail.com|
|{priya@gmail.com, NULL}      |102       |Yashoda           |Not Available|priya@gmail.com|
|{NULL, 9876500013}           |103       |Care              |9876500013   |Not Available  |
|{sneha@gmail.com, 9876500014}|104       |NULL              |9876500014   |sneha@gmail.com|
+-----------------------------+----------+------------------+-------------+---------------+



In [214]:
#80
preferences.join(patients,"patient_id","left").show(truncate=False)

+----------+-----------------------------+------------------+-------------+---------------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|contact                      |preferred_hospital|phone        |email          |patient_name|city     |age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+-----------------------------+------------------+-------------+---------------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|101       |{rahul@gmail.com, 9876500011}|Apollo            |9876500011   |rahul@gmail.com|Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |Complete           |Adult    |1             |0             |
|102       |{priya@gmail.com, NULL}      |Yashoda           |Not Available|priya@gmail.com|Priya Reddy |Bangalore|29 |Female|A+         |Act

**Spark SQL**

In [225]:
#81
patients.createOrReplaceTempView("patients_view")
spark.sql("SELECT * FROM patients_view").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|    Young|             1|             0|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|    Adult|             0|             0|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|    Adult|             1|             0|
|       105|  Farhan Ali|    Delhi| 55|  Male|  

In [226]:
#82
appointments.createOrReplaceTempView("appointments_view")
spark.sql("SELECT * FROM appointments_view").show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|          2000.0|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|          1500.0|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|          2500.0|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|          1000.0|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|          2000.0|Completed|
|          5008|       110|  Dr. Priya|O

In [227]:
#83
spark.sql("SELECT * FROM patients_view").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|    Young|             1|             0|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|    Adult|             0|             0|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|    Adult|             1|             0|
|       105|  Farhan Ali|    Delhi| 55|  Male|  

In [228]:
#84
spark.sql("SELECT * FROM patients_view WHERE city = 'Hyderabad'").show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|    Adult|             1|             0|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|         Incomplete|    Adult|             0|             0|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+



In [230]:
#85
spark.sql("SELECT city, COUNT(*) AS total FROM patients_view GROUP BY city").show()

+---------+-----+
|     city|total|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [231]:
#86
spark.sql("SELECT department, COUNT(*) AS total FROM appointments_view GROUP BY department").show()

+-----------+-----+
| department|total|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [232]:
#87
spark.sql("SELECT department, AVG(consultation_fee) AS avg_fee FROM appointments_view GROUP BY department").show()

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [233]:
#88
spark.sql("SELECT MAX(consultation_fee) as highest_fee FROM appointments_view").show()

+-----------+
|highest_fee|
+-----------+
|     2500.0|
+-----------+



In [234]:
#89
spark.sql("SELECT patient_id, COUNT(*) AS appointments FROM appointments_view GROUP BY patient_id").show()

+----------+------------+
|patient_id|appointments|
+----------+------------+
|       108|           1|
|       101|           2|
|       103|           1|
|       120|           1|
|       107|           1|
|       102|           1|
|       105|           1|
|       110|           1|
|       104|           1|
+----------+------------+



In [248]:
#90
spark.sql("SELECT patient_id, SUM(consultation_fee) AS spending FROM appointments_view GROUP BY patient_id ORDER BY spending DESC LIMIT 5").show()

+----------+--------+
|patient_id|spending|
+----------+--------+
|       101|  2500.0|
|       110|  2500.0|
|       104|  2500.0|
|       107|  2000.0|
|       102|  2000.0|
+----------+--------+



**ETL Project**

In [249]:
#91
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
patients.show()
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
appointments.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [250]:
#92
preferences = spark.read.option("multiline",True).json("patient_preferences.json")
preferences.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [261]:
#93
patients_clean = patients.fillna({"city": "Unknown","blood_group": "Not Available"})
appointments_clean = appointments.fillna({"consultation_fee": 0})
preferences_clean = preferences.fillna({"preferred_hospital": "Unknown"})

patients_clean.show()
appointments_clean.show()
preferences_clean.show(truncate=False)

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [266]:
#94
df = patients_clean.join(appointments_clean,"patient_id","left").join(preferences_clean,"patient_id","left")
df.show(truncate=False)

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+-----------------------------+------------------+
|patient_id|patient_name|city     |age|gender|blood_group  |insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |contact                      |preferred_hospital|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+-----------------------------+------------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+           |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000.0          |Completed|{rahul@gmail.com, 9876500011}|Apollo            |
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+           |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500.0          |Completed|

In [267]:
#95
df.withColumn("age_group",when(col("age")<30, "Young").when(col("age")<50, "Adult").otherwise("Senior")).show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+--------------------+------------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|             contact|preferred_hospital|age_group|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+--------------------+------------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|{rahul@gmail.com,...|            Apollo|    Adult|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Comple

In [269]:
#96
df = df.withColumn("revenue",col("consultation_fee"))
df.select("patient_id","patient_name","consultation_fee","revenue").show()

+----------+------------+----------------+-------+
|patient_id|patient_name|consultation_fee|revenue|
+----------+------------+----------------+-------+
|       101|Rahul Sharma|          1000.0| 1000.0|
|       101|Rahul Sharma|          1500.0| 1500.0|
|       102| Priya Reddy|          2000.0| 2000.0|
|       103|  Amit Kumar|          1500.0| 1500.0|
|       104| Sneha Patel|          2500.0| 2500.0|
|       105|  Farhan Ali|          1000.0| 1000.0|
|       106|  Neha Singh|            NULL|   NULL|
|       107| Arjun Verma|          2000.0| 2000.0|
|       108|  Meera Nair|             0.0|    0.0|
|       109|   Kiran Rao|            NULL|   NULL|
|       110| Nisha Reddy|          2500.0| 2500.0|
+----------+------------+----------------+-------+



In [270]:
#97
spending = df.groupBy("patient_id","patient_name").agg(sum("consultation_fee").alias("spending"))
spending.show()

+----------+------------+--------+
|patient_id|patient_name|spending|
+----------+------------+--------+
|       107| Arjun Verma|  2000.0|
|       108|  Meera Nair|     0.0|
|       109|   Kiran Rao|    NULL|
|       110| Nisha Reddy|  2500.0|
|       105|  Farhan Ali|  1000.0|
|       101|Rahul Sharma|  2500.0|
|       104| Sneha Patel|  2500.0|
|       103|  Amit Kumar|  1500.0|
|       106|  Neha Singh|    NULL|
|       102| Priya Reddy|  2000.0|
+----------+------------+--------+



In [273]:
#98
dept_revenue = df.groupBy("department").agg(sum("consultation_fee").alias("department_revenue"))
dept_revenue.show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|       NULL|              NULL|
|  Neurology|            4000.0|
|Dermatology|            2000.0|
| Cardiology|            3000.0|
|Orthopedics|            5000.0|
+-----------+------------------+



In [274]:
#99
df.write.mode("overwrite").parquet("hospital_analytics")
paraquet_df = spark.read.parquet("hospital_analytics")
paraquet_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+--------------------+------------------+-------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|             contact|preferred_hospital|revenue|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+--------------------+------------------+-------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|          1000.0|Completed|{rahul@gmail.com,...|            Apollo| 1000.0|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|          1500.0|Completed|{rah

In [285]:
#100
print("===== HOSPITAL ANALYTICS REPORT =====")

print("Total Patients :", patients.count())
print("Total Appointments :", appointments.count())
print("Total Revenue :", df.agg(sum("consultation_fee")).collect()[0][0])
print("Average Consultation Fee :", df.agg(avg("consultation_fee")).collect()[0][0])
print("Highest Consultation Fee :", df.agg(max("consultation_fee")).collect()[0][0])
print("Lowest Consultation Fee :", df.agg(min("consultation_fee")).collect()[0][0])
print("Patient Wise Spending")
spending.show()
print("Department Revenue")
dept_revenue.show()

===== HOSPITAL ANALYTICS REPORT =====
Total Patients : 10
Total Appointments : 10
Total Revenue : 14000.0
Average Consultation Fee : 1555.5555555555557
Highest Consultation Fee : 2500.0
Lowest Consultation Fee : 0.0
Patient Wise Spending
+----------+------------+--------+
|patient_id|patient_name|spending|
+----------+------------+--------+
|       107| Arjun Verma|  2000.0|
|       108|  Meera Nair|     0.0|
|       109|   Kiran Rao|    NULL|
|       110| Nisha Reddy|  2500.0|
|       105|  Farhan Ali|  1000.0|
|       101|Rahul Sharma|  2500.0|
|       104| Sneha Patel|  2500.0|
|       103|  Amit Kumar|  1500.0|
|       106|  Neha Singh|    NULL|
|       102| Priya Reddy|  2000.0|
+----------+------------+--------+

Department Revenue
+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|       NULL|              NULL|
|  Neurology|            4000.0|
|Dermatology|            2000.0|
| Cardiology|            3000.0|
|Orthopedics|        